# 04 — Normalizar autores y afiliaciones UNAM

Esta versión toma **`autores_unam_integrados.csv`** como entrada y utiliza:

- `Diccionario_autore.csv`
- `Diccionario_afiliaciones.csv`
- `Afiliaciones_contextuales.csv`
- `Registros_externos_UNAM.csv`

Los archivos se colocan en `04_Limpieza/02_normalizacion/`. La entrada se mantiene en
`04_Limpieza/01_Internos_unam/`; el notebook, en `notebooks/`.

Se conservan el estilo secuencial y los 14 apartados de la versión anterior.
Solo cambian `Autor_norm`, `Afiliacion1` y `Afiliacion2`, o se excluyen relaciones
autor–publicación justificadas. Las otras doce columnas permanecen intactas.
No se intenta reproducir el antiguo número de filas ni las 131 exclusiones históricas.

Los diccionarios son completos y admiten alias no utilizados en esta entrada.
Las claves contextuales se comparan antes de modificar los valores originales.
La afiliación genérica de UNAM se conserva cuando no puede precisarse la dependencia.
Las dos unidades UNAM distintas se mantienen por separado; LaPI no se sustituye por
CECAv, ni Investigaciones Bibliográficas por Ingeniería.

Se incorpora la revisión final de las 29 formas de autor de dos componentes presentes
en el resultado anterior. Solo se amplían los nombres con evidencia suficiente; el
diccionario conserva las fuentes y las búsquedas sin expansión confirmada.
Por instrucción explícita del usuario, `Xavier Góngora` se representa como
`Francisco Xavier Zúñiga Góngora` (no `Francisco Javier Zúñiga Góngora`).
Esta actualización cambia únicamente nombres respecto del resultado normalizado
anterior: no altera afiliaciones, filas, orden ni los otros metadatos.
Las iniciales sin expansión documentada ya fueron tratadas según la regla acordada;
el código no inventa nombres.

Se generan la base normalizada (15 columnas), la copia de filas excluidas y las
auditorías resumen y detallada. Los insumos nunca se sobrescriben. Si existen salidas
anteriores distintas, guárdalas primero y cambia `actualizar_archivos = True`.
La cobertura completa del diccionario es una validación estructural, no una afirmación
de que se hayan reconsultado todas las publicaciones originales.


In [1]:
import os
import re
import csv
import json
import hashlib
import unicodedata
from pathlib import Path
from collections import Counter
import pandas as pd

# ============================================================
# 04 - NORMALIZAR AUTORES Y AFILIACIONES UNAM
# Se reutilizan y corrigen los diccionarios de la fase anterior.
# SOLO cambian Autor_norm, Afiliacion1 y Afiliacion2.
# No se deduplica, renumera, completa ni limpia la bibliografía.
# ============================================================

archivo_entrada = "../04_Limpieza/01_Internos_unam/autores_unam_integrados.csv"
carpeta_salida = "../04_Limpieza/02_normalizacion"

archivo_dic_autores = f"{carpeta_salida}/Diccionario_autore.csv"
archivo_dic_afiliaciones = f"{carpeta_salida}/Diccionario_afiliaciones.csv"
archivo_contextuales = f"{carpeta_salida}/Afiliaciones_contextuales.csv"
archivo_externos = f"{carpeta_salida}/Registros_externos_UNAM.csv"

archivo_salida = f"{carpeta_salida}/autores_unam_normalizados.csv"
archivo_excluidos = f"{carpeta_salida}/registros_excluidos_no_UNAM.csv"
archivo_auditoria = f"{carpeta_salida}/auditoria_normalizacion_autores_afiliaciones.csv"
archivo_detalle = f"{carpeta_salida}/auditoria_detallada_normalizacion.csv"

# False: crea archivos nuevos; si ya existe un resultado diferente, se detiene.
# True: autoriza reemplazar SOLO los cuatro archivos de salida de esta fase.
# Nunca se sobrescriben la entrada ni los diccionarios ni las decisiones.
actualizar_archivos = True

# Opcional: fija la versión de la entrada por su SHA-256.
# La huella SIEMPRE se registra en la auditoría, aunque se deje vacío.
sha256_entrada_esperada = ""

columnas_canonicas = [
    "Fuente_origen", "indice", "Titulo", "Año", "Autor_norm",
    "Afiliacion1", "Afiliacion2", "ISBN", "ISSN", "Doi",
    "URL", "Area", "SubArea", "Keywords", "Abstract"
]
columnas_modificables = ["Autor_norm", "Afiliacion1", "Afiliacion2"]
columnas_inmutables = [
    c for c in columnas_canonicas if c not in columnas_modificables
]

clave_base = [
    "Fuente_origen", "indice", "Titulo", "Año",
    "Autor_norm", "Doi", "Afiliacion1", "Afiliacion2"
]
clave_contexto = [
    "Fuente_origen", "indice", "Titulo", "Año",
    "Autor_actual", "Doi", "Afiliacion1_original", "Afiliacion2_original"
]
esquema_dic_autores = [
    "Autor_actual", "Autor_final", "Origen_nombre", "Aplicar", "Comentario"
]
esquema_dic_afiliaciones = [
    "Afiliacion_actual", "Afiliacion1_final", "Afiliacion2_final",
    "Clasificacion", "Origen", "Aplicar", "Comentario"
]
esquema_contextuales = [
    "Fuente_origen", "indice", "Titulo", "Año", "Autor_actual", "Doi",
    "Afiliacion1_original", "Afiliacion2_original", "Afiliacion1_final",
    "Afiliacion2_final", "Metodo_resolucion", "Evidencia",
    "Candidata_exclusion_no_UNAM"
]
esquema_externos = [
    "Fuente_origen", "indice", "Titulo", "Año", "Autor_actual",
    "Autor_canonico", "Afiliacion1_original", "Afiliacion2_original",
    "Doi", "Evidencia", "Motivo", "Decision"
]
clasificaciones_afiliacion = {
    "UNAM", "UNAM_DOBLE", "GENERICA_UNAM", "EXTERNA", "AMBIGUA"
}
unam_generica = "Universidad Nacional Autónoma de México"
cecav_canonica = "Centro de Estudios en Computación Avanzada"

# ============================================================
# FUNCIONES
# ============================================================

def resolver_ruta(ruta):
    """Admite ejecutar el notebook desde notebooks/ o desde la raíz del repo."""
    p = Path(ruta)
    if p.is_absolute():
        return p
    if p.exists() or Path("..", "04_Limpieza").is_dir():
        return p
    if ruta.startswith("../") and Path("04_Limpieza").is_dir():
        return Path(ruta[3:])
    return p


def leer_csv(ruta):
    ruta = Path(ruta)
    if not ruta.is_file():
        raise FileNotFoundError(f"Falta el archivo: {ruta}")
    with ruta.open("r", encoding="utf-8-sig", newline="") as f:
        inicio = f.read(150)
    if inicio.startswith("version https://git-lfs.github.com/spec/v1"):
        raise ValueError(
            f"{ruta.name} es un puntero Git LFS, no el CSV con datos. "
            "Descarga el contenido real antes de continuar."
        )
    return pd.read_csv(
        ruta, dtype=str, keep_default_na=False, encoding="utf-8-sig"
    )


def validar_columnas(df, columnas, nombre):
    if list(df.columns) != columnas:
        raise ValueError(
            f"{nombre}: esquema inesperado.\n"
            f"Esperadas: {columnas}\nEncontradas: {list(df.columns)}"
        )
    if any(c.startswith("Unnamed") for c in df.columns):
        raise ValueError(f"{nombre}: contiene columnas Unnamed.")
    if not all(isinstance(v, str) for v in df.to_numpy().ravel()):
        raise ValueError(f"{nombre}: se requieren todos los valores como texto.")


def es_nfc(texto):
    return unicodedata.normalize("NFC", texto) == texto


def crear_claves(df, columnas):
    # Las claves usan los valores ORIGINALES: no se borran acentos,
    # no se convierte el año y no se cambia la forma del nombre.
    return pd.Series(
        list(df[columnas].itertuples(index=False, name=None)),
        index=df.index, dtype="object"
    )


def huella_archivo(ruta):
    return hashlib.sha256(Path(ruta).read_bytes()).hexdigest()


def huella_fila(fila):
    contenido = json.dumps(
        [fila[c] for c in columnas_canonicas],
        ensure_ascii=False, separators=(",", ":")
    )
    return hashlib.sha256(contenido.encode("utf-8")).hexdigest()


def valores_finales_afiliacion(fila):
    valores = []
    for campo in ["Afiliacion1_final", "Afiliacion2_final"]:
        valor = fila[campo]
        if valor and valor not in valores:
            valores.append(valor)
    return valores


def agregar_sin_repetir(destino, valores):
    for valor in valores:
        if valor and valor not in destino:
            destino.append(valor)


def quitar_unam_generica_redundante(valores):
    # Universidad + dependencia de esa misma universidad no son
    # dos adscripciones independientes. No se elimina otra unidad.
    if len(valores) > 1 and unam_generica in valores:
        return [v for v in valores if v != unam_generica]
    return valores


def validar_cecav(valor):
    # LaPI NO es un alias de CECAv.
    s = valor.casefold()
    if ("centro de estudios" in s
            and ("computacion avanzada" in s or "computación avanzada" in s)
            and valor != cecav_canonica):
        raise ValueError(f"Denominación CECAv no canónica: {valor!r}")


def detectar_cientificos(df):
    patron = re.compile(r"^[+-]?\d+(?:[.,]\d+)?[eE][+-]?\d+$")
    casos = []
    for campo in ["indice", "ISBN", "ISSN", "Doi"]:
        for idx, valor in df[campo].items():
            for token in valor.split(";"):
                if patron.fullmatch(token.strip()):
                    casos.append((idx + 1, campo, token))
    return casos


def comprobar_relectura(df, ruta):
    recuperado = leer_csv(ruta)
    if list(recuperado.columns) != list(df.columns):
        raise ValueError(f"{Path(ruta).name}: cambió el esquema al guardar.")
    if recuperado.shape != df.shape:
        raise ValueError(f"{Path(ruta).name}: cambiaron las dimensiones.")
    if not recuperado.equals(df.reset_index(drop=True)):
        raise ValueError(f"{Path(ruta).name}: cambió al menos una celda.")
    if any(c.startswith("Unnamed") for c in recuperado.columns):
        raise ValueError(f"{Path(ruta).name}: aparecieron columnas Unnamed.")
    return recuperado


def guardar_csv_verificado(df, ruta):
    """Escritura temporal y relectura antes de sustituir SOLO una salida."""
    ruta = Path(ruta)
    if ruta.exists():
        existente = leer_csv(ruta)
        if existente.equals(df.reset_index(drop=True)):
            comprobar_relectura(df, ruta)
            return
        if not actualizar_archivos:
            raise FileExistsError(f"No se autorizó reemplazar {ruta}.")
    temporal = ruta.with_name(ruta.stem + ".__tmp__.csv")
    try:
        df.to_csv(
            temporal, index=False, encoding="utf-8-sig",
            quoting=csv.QUOTE_ALL, lineterminator="\n"
        )
        comprobar_relectura(df, temporal)
        os.replace(temporal, ruta)
        comprobar_relectura(df, ruta)
    finally:
        if temporal.exists():
            temporal.unlink()


# ============================================================
# 1. CARGA
# ============================================================

archivo_entrada = resolver_ruta(archivo_entrada)
archivo_dic_autores = resolver_ruta(archivo_dic_autores)
archivo_dic_afiliaciones = resolver_ruta(archivo_dic_afiliaciones)
archivo_contextuales = resolver_ruta(archivo_contextuales)
archivo_externos = resolver_ruta(archivo_externos)
archivo_salida = resolver_ruta(archivo_salida)
archivo_excluidos = resolver_ruta(archivo_excluidos)
archivo_auditoria = resolver_ruta(archivo_auditoria)
archivo_detalle = resolver_ruta(archivo_detalle)

insumos = [
    archivo_entrada, archivo_dic_autores, archivo_dic_afiliaciones,
    archivo_contextuales, archivo_externos
]
huellas_antes = {str(p): huella_archivo(p) for p in insumos}
if (sha256_entrada_esperada
        and huellas_antes[str(archivo_entrada)] != sha256_entrada_esperada):
    raise ValueError("La entrada no coincide con la versión SHA-256 esperada.")

entrada = leer_csv(archivo_entrada)
dic_autores = leer_csv(archivo_dic_autores)
dic_afiliaciones = leer_csv(archivo_dic_afiliaciones)
contextuales = leer_csv(archivo_contextuales)
externos = leer_csv(archivo_externos)

validar_columnas(entrada, columnas_canonicas, archivo_entrada.name)
validar_columnas(dic_autores, esquema_dic_autores, archivo_dic_autores.name)
validar_columnas(dic_afiliaciones, esquema_dic_afiliaciones, archivo_dic_afiliaciones.name)
validar_columnas(contextuales, esquema_contextuales, archivo_contextuales.name)
validar_columnas(externos, esquema_externos, archivo_externos.name)
entrada_original = entrada.copy(deep=True)

print("=== CARGA ===")
print("Filas de entrada:", len(entrada))
print("Columnas:", len(entrada.columns))
print("Variantes de autor del diccionario:", len(dic_autores))
print("Afiliaciones del diccionario:", len(dic_afiliaciones))
print("Resoluciones contextuales:", len(contextuales))
print("Exclusiones contextuales explícitas:", len(externos))
print()

# ============================================================
# 2. VALIDACIONES DE LA ENTRADA
# ============================================================

if entrada.empty:
    raise ValueError("La entrada no contiene registros.")
if entrada["Autor_norm"].str.strip().eq("").any():
    raise ValueError("Hay autores vacíos; resolver la separación antes de normalizar.")
if entrada["Autor_norm"].str.contains(r"[;|]", regex=True).any():
    raise ValueError("Una celda Autor_norm parece contener varios autores.")
if (entrada["Afiliacion1"].eq("") & entrada["Afiliacion2"].eq("")).any():
    raise ValueError("Hay registros sin ninguna afiliación. No se inventará UNAM.")
if entrada["indice"].eq("").any():
    raise ValueError("Hay índices vacíos.")
cientificos = detectar_cientificos(entrada)
if cientificos:
    raise ValueError(
        "Hay identificadores en notación científica; no se reconstruyen "
        f"en normalización. Primeros casos: {cientificos[:10]}"
    )

# No se exige NFC en la entrada: el diccionario contiene sus variantes
# originales y entrega las formas finales NFC.
claves_base = crear_claves(entrada, clave_base)
conteo_claves_base = Counter(claves_base)
autores_base = set(entrada["Autor_norm"])
afiliaciones_base = set(entrada["Afiliacion1"]) | set(entrada["Afiliacion2"])
afiliaciones_base.discard("")

# ============================================================
# 3. VALIDAR EXCLUSIONES DEFINITIVAS
# ============================================================

claves_externos = crear_claves(externos, clave_contexto)
if claves_externos.duplicated().any():
    raise ValueError("Hay decisiones de exclusión duplicadas.")
if not set(externos["Decision"]).issubset({"EXCLUIR"}):
    raise ValueError("Las exclusiones deben estar decididas explícitamente como EXCLUIR.")
for campo in ["Evidencia", "Motivo"]:
    if externos[campo].str.strip().eq("").any():
        raise ValueError(f"Hay exclusiones sin {campo}.")
faltan_externos = [k for k in claves_externos if k not in conteo_claves_base]
if faltan_externos:
    raise ValueError(
        f"{len(faltan_externos)} exclusiones no corresponden a esta entrada. "
        "No se aplican por nombre, apellido o índice solamente."
    )
mapa_exclusiones = {
    tuple(f[c] for c in clave_contexto): f.to_dict()
    for _, f in externos.iterrows()
}

# ============================================================
# 4. VALIDAR DICCIONARIO DE AUTORES
# ============================================================

if dic_autores["Autor_actual"].duplicated().any():
    raise ValueError("Autor_actual no es una clave única.")
if dic_autores["Autor_final"].str.strip().eq("").any():
    raise ValueError("El diccionario contiene nombres finales vacíos.")
if not set(dic_autores["Aplicar"]).issubset({"SI", "NO"}):
    raise ValueError("Aplicar de autores solo admite SI o NO.")
faltan_autores = autores_base - set(dic_autores["Autor_actual"])
if faltan_autores:
    raise ValueError(
        "Faltan variantes de autor en el diccionario; no habrá fuzzy matching "
        f"ni expansión de iniciales automática: {sorted(faltan_autores)}"
    )
for _, fila in dic_autores.iterrows():
    final = fila["Autor_final"]
    if not es_nfc(final) or final != final.strip():
        raise ValueError(f"Autor_final no NFC o con espacios terminales: {final!r}")
    if re.search(r"[,;|]|[-‐‑‒–—−]", final):
        raise ValueError(f"Quedó puntuación estructural no canónica: {final!r}")
    if re.search(r"\b[A-ZÁÉÍÓÚÜÑ]\.", final):
        raise ValueError(f"Quedó una inicial con punto sin resolver: {final!r}")
    if fila["Aplicar"] == "NO" and final != fila["Autor_actual"]:
        raise ValueError(
            f"Aplicar=NO no es una exclusión: exige conservar el nombre: {fila['Autor_actual']!r}"
        )
dic_autores_idx = dic_autores.set_index("Autor_actual")
# Se permite que el diccionario tenga alias no utilizados en esta ejecución.
autores_dic_no_usados = set(dic_autores["Autor_actual"]) - autores_base

# ============================================================
# 5. VALIDAR DICCIONARIO DE AFILIACIONES
# ============================================================

if dic_afiliaciones["Afiliacion_actual"].duplicated().any():
    raise ValueError("Afiliacion_actual no es una clave única.")
if not set(dic_afiliaciones["Clasificacion"]).issubset(clasificaciones_afiliacion):
    raise ValueError("Hay clasificaciones institucionales desconocidas.")
if not set(dic_afiliaciones["Aplicar"]).issubset({"SI", "NO", "REVISAR"}):
    raise ValueError("Hay estados Aplicar de afiliación desconocidos.")
faltan_afiliaciones = afiliaciones_base - set(dic_afiliaciones["Afiliacion_actual"])
if faltan_afiliaciones:
    raise ValueError(
        "Faltan afiliaciones en el diccionario. No se asignará una facultad "
        f"por la palabra UNAM ni por la afiliación de otro autor: {sorted(faltan_afiliaciones)}"
    )
for _, fila in dic_afiliaciones.iterrows():
    valores = valores_finales_afiliacion(fila)
    clasificacion = fila["Clasificacion"]
    aplicar = fila["Aplicar"]
    if fila["Afiliacion2_final"] and not fila["Afiliacion1_final"]:
        raise ValueError("Afiliacion2_final no puede estar sola.")
    if (fila["Afiliacion2_final"]
            and fila["Afiliacion1_final"] == fila["Afiliacion2_final"]):
        raise ValueError("El diccionario duplica una afiliación en ambas columnas.")
    for valor in valores:
        if not es_nfc(valor) or valor != valor.strip():
            raise ValueError(f"Afiliación final no NFC o con espacios terminales: {valor!r}")
        validar_cecav(valor)
    if aplicar == "NO":
        if (fila["Afiliacion1_final"] != fila["Afiliacion_actual"]
                or fila["Afiliacion2_final"]):
            raise ValueError("Aplicar=NO exige conservar exactamente la cadena.")
    if clasificacion == "EXTERNA" and valores:
        raise ValueError("Una institución externa no puede producir una afiliación UNAM.")
    if clasificacion == "AMBIGUA" and aplicar != "REVISAR":
        raise ValueError("Una afiliación AMBIGUA requiere resolución contextual.")
    if aplicar != "REVISAR":
        if clasificacion not in {"EXTERNA", "AMBIGUA"} and not valores:
            raise ValueError("Una afiliación UNAM resuelta debe tener salida.")
        if clasificacion == "UNAM_DOBLE" and len(valores) != 2:
            raise ValueError("UNAM_DOBLE debe tener dos salidas distintas.")
        if clasificacion == "GENERICA_UNAM" and valores != [unam_generica]:
            raise ValueError("GENERICA_UNAM no puede inventar una dependencia.")

dic_afiliaciones_idx = dic_afiliaciones.set_index("Afiliacion_actual")
catalogo_final_unam = set()
for _, fila in dic_afiliaciones.iterrows():
    if fila["Clasificacion"] not in {"EXTERNA", "AMBIGUA"}:
        catalogo_final_unam.update(valores_finales_afiliacion(fila))
afiliaciones_dic_no_usadas = set(dic_afiliaciones["Afiliacion_actual"]) - afiliaciones_base

# ============================================================
# 6. VALIDAR RESOLUCIONES CONTEXTUALES
# ============================================================

claves_contextuales = crear_claves(contextuales, clave_contexto)
if claves_contextuales.duplicated().any():
    raise ValueError("Hay decisiones contextuales duplicadas o contradictorias.")
if set(claves_contextuales) & set(claves_externos):
    raise ValueError("Una misma relación figura a la vez como conservada y excluida.")
faltan_contextos = [k for k in claves_contextuales if k not in conteo_claves_base]
if faltan_contextos:
    raise ValueError(
        f"{len(faltan_contextos)} resoluciones no coinciden con la entrada actual. "
        "No se recuperan por un apellido o una posición histórica."
    )
for _, fila in contextuales.iterrows():
    vals = valores_finales_afiliacion(fila)
    if not vals or not fila["Afiliacion1_final"]:
        raise ValueError("Hay una resolución contextual sin afiliación final.")
    if (fila["Afiliacion2_final"]
            and fila["Afiliacion1_final"] == fila["Afiliacion2_final"]):
        raise ValueError("Una resolución contextual repite la misma afiliación.")
    if set(vals) - catalogo_final_unam:
        raise ValueError("La resolución usa una denominación que falta en el catálogo UNAM.")
    if any(not es_nfc(v) or v != v.strip() for v in vals):
        raise ValueError("Una resolución contextual no tiene la forma final NFC.")
    if not fila["Evidencia"].strip() or not fila["Metodo_resolucion"].strip():
        raise ValueError("Una resolución contextual carece de método o evidencia.")
    if fila["Candidata_exclusion_no_UNAM"] != "NO":
        raise ValueError("Una candidata a exclusión debe resolverse antes de aplicar el contexto.")

mapa_contextual = {
    tuple(f[c] for c in clave_contexto): f.to_dict()
    for _, f in contextuales.iterrows()
}
for idx, fila in entrada.iterrows():
    clave = claves_base.at[idx]
    if clave in mapa_contextual or clave in mapa_exclusiones:
        continue
    for campo in ["Afiliacion1", "Afiliacion2"]:
        original = fila[campo]
        if original and dic_afiliaciones_idx.at[original, "Aplicar"] == "REVISAR":
            raise ValueError(
                f"Registro de datos {idx+1}: afiliación REVISAR sin resolución contextual."
            )
print("Validaciones previas: OK\n")

# ============================================================
# 7. EXCLUIR REGISTROS DEFINITIVAMENTE EXTERNOS
# ============================================================

motivos_exclusion = {}
evidencias_exclusion = {}
for idx, fila in entrada.iterrows():
    clave = claves_base.at[idx]
    if clave in mapa_exclusiones:
        decision = mapa_exclusiones[clave]
        motivos_exclusion[idx] = decision["Motivo"]
        evidencias_exclusion[idx] = decision["Evidencia"]
        continue
    if clave in mapa_contextual:
        continue
    decisiones = [
        dic_afiliaciones_idx.loc[fila[campo]]
        for campo in ["Afiliacion1", "Afiliacion2"] if fila[campo]
    ]
    if decisiones and all(d["Clasificacion"] == "EXTERNA" for d in decisiones):
        # Esta exclusión corresponde a la relación con esas afiliaciones,
        # nunca a todas las publicaciones de la persona.
        motivos_exclusion[idx] = "Las afiliaciones de esta relación están clasificadas como externas."
        evidencias_exclusion[idx] = " | ".join(
            d["Comentario"] for d in decisiones
        )

indices_excluidos = sorted(motivos_exclusion)
registros_excluidos = entrada.loc[indices_excluidos, columnas_canonicas].copy()
trabajo = entrada.drop(index=indices_excluidos).copy()
if trabajo.empty:
    raise ValueError("Ninguna relación UNAM sobrevivió; revisar la entrada antes de publicar una base vacía.")
print("=== EXCLUSIÓN CONTEXTUAL ===")
print("Filas retiradas:", len(registros_excluidos))
print("Filas que continúan:", len(trabajo))
print()

# ============================================================
# 8. NORMALIZAR AUTORES
# ============================================================

trabajo["Autor_norm"] = [
    dic_autores_idx.at[a, "Autor_final"] for a in trabajo["Autor_norm"]
]
# Eliminar iniciales o elegir una forma completa ya está decidido
# en el diccionario. Aquí no se expande ni se infiere identidad.

# ============================================================
# 9. PREPARAR RESOLUCIONES DE AFILIACIONES
# ============================================================

metodos_afiliacion = {}
evidencias_afiliacion = {}
externas_retiradas = Counter()
genericas_redundantes = set()
afiliaciones_repetidas_resueltas = set()

# ============================================================
# 10. NORMALIZAR AFILIACIONES
# ============================================================

for idx in trabajo.index:
    fila = entrada.loc[idx]
    clave = claves_base.at[idx]
    if clave in mapa_contextual:
        decision = mapa_contextual[clave]
        finales = valores_finales_afiliacion(decision)
        previas = []
        for campo in ["Afiliacion1", "Afiliacion2"]:
            if fila[campo]:
                d_previa = dic_afiliaciones_idx.loc[fila[campo]]
                agregar_sin_repetir(previas, valores_finales_afiliacion(d_previa))
        if (len(previas) > 1 and unam_generica in previas
                and unam_generica not in finales):
            genericas_redundantes.add(idx)
        metodos_afiliacion[idx] = "CONTEXTUAL: " + decision["Metodo_resolucion"]
        evidencias_afiliacion[idx] = decision["Evidencia"]
    else:
        finales = []
        trazas = []
        for campo in ["Afiliacion1", "Afiliacion2"]:
            original = fila[campo]
            if not original:
                continue
            decision = dic_afiliaciones_idx.loc[original]
            if decision["Aplicar"] == "REVISAR":
                raise ValueError("Se intentó aplicar una afiliación pendiente.")
            vals = valores_finales_afiliacion(decision)
            if decision["Clasificacion"] == "EXTERNA":
                externas_retiradas[idx] += 1
            if set(vals) & set(finales):
                afiliaciones_repetidas_resueltas.add(idx)
            agregar_sin_repetir(finales, vals)
            trazas.append(
                campo + ": " + decision["Origen"] + " | " + decision["Comentario"]
            )
        metodos_afiliacion[idx] = "DICCIONARIO"
        evidencias_afiliacion[idx] = " || ".join(trazas)
    if len(finales) > 1 and unam_generica in finales:
        genericas_redundantes.add(idx)
        finales = quitar_unam_generica_redundante(finales)
        evidencias_afiliacion[idx] += (
            " | Se retira la universidad genérica redundante frente a su dependencia explícita."
        )
    if not finales:
        raise ValueError(f"El registro {idx+1} quedó sin afiliación UNAM.")
    if len(finales) > 2:
        raise ValueError(
            f"El registro {idx+1} produciría más de dos afiliaciones UNAM. "
            "No se truncan: requiere una resolución contextual."
        )
    trabajo.at[idx, "Afiliacion1"] = finales[0]
    trabajo.at[idx, "Afiliacion2"] = finales[1] if len(finales) == 2 else ""

# ============================================================
# 11. VALIDACIONES FINALES
# ============================================================

validar_columnas(trabajo, columnas_canonicas, "salida en memoria")
if len(trabajo) + len(registros_excluidos) != len(entrada):
    raise ValueError("No se contabilizaron todas las filas de entrada.")
if set(trabajo.index) & set(registros_excluidos.index):
    raise ValueError("Una fila quedó a la vez conservada y excluida.")
if list(trabajo.index) != sorted(trabajo.index):
    raise ValueError("Se alteró el orden relativo de los registros.")
for columna in columnas_inmutables:
    if not trabajo[columna].equals(entrada.loc[trabajo.index, columna]):
        raise ValueError(f"Se modificó la columna protegida {columna}.")
if not entrada.equals(entrada_original):
    raise ValueError("Se modificó la copia de entrada durante el proceso.")

for columna in columnas_modificables:
    if not trabajo[columna].map(es_nfc).all():
        raise ValueError(f"La salida contiene valores no NFC en {columna}.")
if trabajo["Autor_norm"].str.strip().eq("").any() or trabajo["Afiliacion1"].eq("").any():
    raise ValueError("Hay autores o primeras afiliaciones vacías.")
if (trabajo["Afiliacion2"].ne("") & trabajo["Afiliacion1"].eq(trabajo["Afiliacion2"])).any():
    raise ValueError("Quedaron afiliaciones idénticas en las dos columnas.")
for campo in ["Afiliacion1", "Afiliacion2"]:
    if set(trabajo[campo]) - catalogo_final_unam - {""}:
        raise ValueError("Hay una afiliación final fuera del catálogo revisado.")
    for valor in trabajo[campo]:
        validar_cecav(valor)
if detectar_cientificos(trabajo):
    raise ValueError("Apareció un identificador en notación científica.")
for ruta, huella in huellas_antes.items():
    if huella_archivo(ruta) != huella:
        raise ValueError(f"Se modificó un insumo: {ruta}")
print("Validaciones finales: OK\n")

# ============================================================
# 12. AUDITORÍA
# ============================================================

detalle_filas = []
for idx, fila in entrada.iterrows():
    da = dic_autores_idx.loc[fila["Autor_norm"]]
    conservada = idx in trabajo.index
    final = trabajo.loc[idx] if conservada else None
    cambios = [
        c for c in columnas_modificables if conservada and fila[c] != final[c]
    ]
    detalle_filas.append({
        "Fila_entrada": str(idx + 1),
        "Hash_fila_entrada": huella_fila(fila),
        "Fuente_origen": fila["Fuente_origen"],
        "indice": fila["indice"],
        "Titulo": fila["Titulo"],
        "Año": fila["Año"],
        "Doi": fila["Doi"],
        "Autor_original": fila["Autor_norm"],
        "Autor_final": final["Autor_norm"] if conservada else "",
        "Afiliacion1_original": fila["Afiliacion1"],
        "Afiliacion2_original": fila["Afiliacion2"],
        "Afiliacion1_final": final["Afiliacion1"] if conservada else "",
        "Afiliacion2_final": final["Afiliacion2"] if conservada else "",
        "Decision": "CONSERVAR" if conservada else "EXCLUIR",
        "Metodo_autor": da["Origen_nombre"],
        "Metodo_afiliaciones": metodos_afiliacion.get(idx, "EXCLUSION_CONTEXTUAL"),
        "Campos_modificados": "; ".join(cambios),
        "Evidencia_autor": da["Comentario"],
        "Evidencia_afiliaciones": (
            evidencias_afiliacion.get(idx, evidencias_exclusion.get(idx, ""))
        ),
        "Motivo_exclusion": motivos_exclusion.get(idx, ""),
    })
detalle = pd.DataFrame(detalle_filas)
entrada_conservada = entrada.loc[trabajo.index]
metricas = [
    ("Filas_entrada", len(entrada)),
    ("Columnas_entrada", len(entrada.columns)),
    ("Variantes_autor_entrada", len(autores_base)),
    ("Variantes_afiliacion_entrada", len(afiliaciones_base)),
    ("Variantes_autor_diccionario", len(dic_autores)),
    ("Variantes_afiliacion_diccionario", len(dic_afiliaciones)),
    ("Variantes_autor_sin_diccionario", len(faltan_autores)),
    ("Variantes_afiliacion_sin_diccionario", len(faltan_afiliaciones)),
    ("Alias_autor_no_usados", len(autores_dic_no_usados)),
    ("Alias_afiliacion_no_usados", len(afiliaciones_dic_no_usadas)),
    ("Resoluciones_contextuales", len(contextuales)),
    ("Exclusiones_contextuales_explicitas", len(externos)),
    ("Registros_externos_eliminados", len(registros_excluidos)),
    ("Filas_salida", len(trabajo)),
    ("Columnas_salida", len(trabajo.columns)),
    ("Filas_Autor_norm_modificado", int((entrada_conservada["Autor_norm"] != trabajo["Autor_norm"]).sum())),
    ("Filas_Afiliacion1_modificada", int((entrada_conservada["Afiliacion1"] != trabajo["Afiliacion1"]).sum())),
    ("Filas_Afiliacion2_modificada", int((entrada_conservada["Afiliacion2"] != trabajo["Afiliacion2"]).sum())),
    ("Filas_con_dos_afiliaciones_UNAM", int(trabajo["Afiliacion2"].ne("").sum())),
    ("Filas_solo_UNAM_generica", int((trabajo["Afiliacion1"].eq(unam_generica) & trabajo["Afiliacion2"].eq("")).sum())),
    ("Filas_con_universidad_generica_redundante_retirada", len(genericas_redundantes)),
    ("Filas_con_etiquetas_equivalentes_repetidas", len(afiliaciones_repetidas_resueltas)),
    ("Afiliaciones_EXTERNAS_retiradas_por_diccionario", sum(externas_retiradas.values())),
    ("Filas_contextuales_retiro_mencion_externa", sum("RETIRO_MENCION_EXTERNA" in x for x in metodos_afiliacion.values())),
    ("Filas_afiliacion_contextual", sum(x.startswith("CONTEXTUAL:") for x in metodos_afiliacion.values())),
    ("Formas_autor_finales_distintas", trabajo["Autor_norm"].nunique()),
    ("Afiliaciones_UNAM_finales_distintas", len((set(trabajo["Afiliacion1"]) | set(trabajo["Afiliacion2"])) - {""})),
    ("Columnas_bibliograficas_inmutables", len(columnas_inmutables)),
    ("Duplicados_exactos_entrada_no_eliminados", int(entrada.duplicated().sum())),
    ("Duplicados_exactos_salida_no_eliminados", int(trabajo.duplicated().sum())),
    ("Filas_con_Año_decimal_preservado", int(trabajo["Año"].str.fullmatch(r"202[45]\.0").sum())),
    ("Filas_con_Año_vacio_preservado", int(trabajo["Año"].eq("").sum())),
    ("SHA256_entrada", huellas_antes[str(archivo_entrada)]),
    ("SHA256_diccionario_autores", huellas_antes[str(archivo_dic_autores)]),
    ("SHA256_diccionario_afiliaciones", huellas_antes[str(archivo_dic_afiliaciones)]),
    ("SHA256_contextuales", huellas_antes[str(archivo_contextuales)]),
    ("SHA256_exclusiones", huellas_antes[str(archivo_externos)]),
]
auditoria = pd.DataFrame(
    [(nombre, str(valor)) for nombre, valor in metricas],
    columns=["Metrica", "Valor"]
)

# ============================================================
# 13. GUARDAR
# ============================================================

salida = trabajo[columnas_canonicas].reset_index(drop=True).copy()
excluidos_salida = registros_excluidos[columnas_canonicas].reset_index(drop=True).copy()
productos = [
    (archivo_salida, salida),
    (archivo_excluidos, excluidos_salida),
    (archivo_auditoria, auditoria),
    (archivo_detalle, detalle),
]
# Prever todos los conflictos ANTES de escribir cualquier resultado.
rutas_insumos = {Path(p).resolve() for p in insumos}
for ruta, df_resultado in productos:
    if Path(ruta).resolve() in rutas_insumos:
        raise ValueError("Una salida intenta sobrescribir un insumo.")
    if Path(ruta).exists() and not actualizar_archivos:
        previo = leer_csv(ruta)
        if not previo.equals(df_resultado):
            raise FileExistsError(
                f"Ya existe una salida distinta: {ruta}. "
                "Resguarda la versión anterior y establece actualizar_archivos = True "
                "para autorizar únicamente su reemplazo."
            )
for ruta, df_resultado in productos:
    Path(ruta).parent.mkdir(parents=True, exist_ok=True)
    guardar_csv_verificado(df_resultado, ruta)
verificacion = comprobar_relectura(salida, archivo_salida)
validar_columnas(verificacion, columnas_canonicas, "CSV final releído")
if detectar_cientificos(verificacion):
    raise ValueError("La relectura contiene identificadores científicos.")
for ruta, huella in huellas_antes.items():
    if huella_archivo(ruta) != huella:
        raise ValueError(f"Se modificó un insumo al guardar: {ruta}")

# ============================================================
# 14. RESUMEN
# ============================================================

print("=== RESULTADO ===")
print("Entrada:", archivo_entrada)
print("Archivo normalizado:", archivo_salida)
print("Registros excluidos:", archivo_excluidos)
print("Auditoría resumen:", archivo_auditoria)
print("Auditoría detallada:", archivo_detalle)
print()
print("Filas entrada:", len(entrada))
print("Filas excluidas:", len(registros_excluidos))
print("Filas salida:", len(salida))
print("Columnas:", len(salida.columns))
print("Formas de autor finales:", salida["Autor_norm"].nunique())
print("Filas con dos afiliaciones UNAM:", int(salida["Afiliacion2"].ne("").sum()))
print("Filas con UNAM genérica:", int(
    (salida["Afiliacion1"].eq(unam_generica) & salida["Afiliacion2"].eq("")).sum()
))
print("Cobertura de ambos diccionarios:", "100 % de las variantes de la entrada")
print("Columnas bibliográficas/índice/procedencia intactas:", len(columnas_inmutables))
print("Relectura exacta de todos los CSV:", True)
print("SHA-256 de la salida:", huella_archivo(archivo_salida))
print()
print(
    "Normalización terminada. No se deduplicó ni se limpiaron años, "
    "identificadores, títulos, keywords o abstracts."
)


=== CARGA ===
Filas de entrada: 5106
Columnas: 15
Variantes de autor del diccionario: 1403
Afiliaciones del diccionario: 1144
Resoluciones contextuales: 417
Exclusiones contextuales explícitas: 0

Validaciones previas: OK

=== EXCLUSIÓN CONTEXTUAL ===
Filas retiradas: 0
Filas que continúan: 5106

Validaciones finales: OK

=== RESULTADO ===
Entrada: ..\04_Limpieza\01_Internos_unam\autores_unam_integrados.csv
Archivo normalizado: ..\04_Limpieza\02_normalizacion\autores_unam_normalizados.csv
Registros excluidos: ..\04_Limpieza\02_normalizacion\registros_excluidos_no_UNAM.csv
Auditoría resumen: ..\04_Limpieza\02_normalizacion\auditoria_normalizacion_autores_afiliaciones.csv
Auditoría detallada: ..\04_Limpieza\02_normalizacion\auditoria_detallada_normalizacion.csv

Filas entrada: 5106
Filas excluidas: 0
Filas salida: 5106
Columnas: 15
Formas de autor finales: 669
Filas con dos afiliaciones UNAM: 351
Filas con UNAM genérica: 610
Cobertura de ambos diccionarios: 100 % de las variantes de la e